# Tổng hợp và so sánh KNN, SVM, Decision Tree

## Mini Capstone Machine Learning

Notebook này **không huấn luyện hay tuning lại** bất kỳ mô hình nào. Notebook chỉ đọc kết quả đã được lưu bởi notebook 03 (KNN), 04 (SVM) và 05 (Decision Tree), kiểm tra tính nhất quán của tập test, sau đó tạo bảng và biểu đồ phục vụ báo cáo.

## Nguyên tắc so sánh công bằng

- Cả ba mô hình phải dự đoán đúng cùng **114 hàng test** do notebook 02 tạo ra.
- Siêu tham số của mỗi mô hình đã được chọn bằng cross-validation trên train; kết quả test không được dùng để tuning lại.
- Các chỉ số chính gồm Accuracy, Precision/Recall/F1 của lớp ác tính `M`, F1-macro và ROC-AUC. Trong ngữ cảnh bài toán này, cần đặc biệt chú ý **Recall lớp M** và số ca ác tính bị bỏ sót (False Negative).
- Kết quả có giá trị trong phạm vi dataset và quy trình thực nghiệm này; đây không phải công cụ chẩn đoán y khoa thực tế.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import auc, confusion_matrix, roc_curve
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

outputs_root = project_root / 'outputs'
comparison_dir = outputs_root / 'comparison'
comparison_dir.mkdir(parents=True, exist_ok=True)

model_info = {
    'svm': {'name': 'SVM', 'folder': 'svm'},
    'knn': {'name': 'KNN', 'folder': 'knn'},
    'decision_tree': {'name': 'Decision Tree', 'folder': 'decision_tree'},
}
model_order = ['svm', 'knn', 'decision_tree']

print(f'Thư mục gốc: {project_root}')
print(f'Nơi lưu kết quả tổng hợp: {comparison_dir}')

## 1. Kiểm tra và nạp kết quả của ba mô hình

Cell dưới đây dừng ngay nếu thiếu tệp, số hàng test không bằng 114, hoặc ba mô hình không dự đoán cùng các dòng và nhãn thật. Kiểm tra này tránh việc so sánh các thí nghiệm không công bằng.

In [ ]:
required_files = [
    'baseline_cv_metrics.json', 'best_parameters.json', 'classification_report.csv',
    'confusion_matrix.png', 'cv_results.csv', 'roc_curve.png',
    'test_metrics.json', 'test_predictions.csv',
]
required_prediction_columns = {'row_index', 'actual_diagnosis', 'predicted_diagnosis', 'probability_malignant'}
loaded_results = {}
reference_test_rows = None

for model_key in model_order:
    model_dir = outputs_root / model_info[model_key]['folder']
    missing_files = [file_name for file_name in required_files if not (model_dir / file_name).is_file()]
    if missing_files:
        raise FileNotFoundError(f"{model_info[model_key]['name']} thiếu tệp: {', '.join(missing_files)}")

    predictions = pd.read_csv(model_dir / 'test_predictions.csv')
    missing_columns = required_prediction_columns - set(predictions.columns)
    if missing_columns:
        raise ValueError(f"{model_info[model_key]['name']} thiếu cột: {sorted(missing_columns)}")
    if len(predictions) != 114:
        raise ValueError(f"{model_info[model_key]['name']} có {len(predictions)} dòng test, cần đúng 114 dòng.")
    if predictions['row_index'].duplicated().any():
        raise ValueError(f"{model_info[model_key]['name']} có row_index bị lặp.")
    if not set(predictions['actual_diagnosis']).issubset({'B', 'M'}):
        raise ValueError(f"{model_info[model_key]['name']} có nhãn thật không hợp lệ.")
    if not set(predictions['predicted_diagnosis']).issubset({'B', 'M'}):
        raise ValueError(f"{model_info[model_key]['name']} có nhãn dự đoán không hợp lệ.")

    current_test_rows = predictions[['row_index', 'actual_diagnosis']].copy()
    if reference_test_rows is None:
        reference_test_rows = current_test_rows
    elif not current_test_rows.equals(reference_test_rows):
        raise ValueError(
            f"{model_info[model_key]['name']} không dùng cùng row_index/nhãn thật với các mô hình còn lại."
        )

    with open(model_dir / 'test_metrics.json', encoding='utf-8') as file:
        test_metrics = json.load(file)
    with open(model_dir / 'best_parameters.json', encoding='utf-8') as file:
        best_parameters = json.load(file)
    with open(model_dir / 'baseline_cv_metrics.json', encoding='utf-8') as file:
        baseline_cv_metrics = json.load(file)

    loaded_results[model_key] = {
        'predictions': predictions,
        'test_metrics': test_metrics,
        'best_parameters': best_parameters,
        'baseline_cv_metrics': baseline_cv_metrics,
        'cv_results': pd.read_csv(model_dir / 'cv_results.csv'),
    }

print(f'Đã kiểm tra thành công: {len(reference_test_rows)} mẫu test giống nhau cho cả ba mô hình.')

## 2. Bảng cấu hình tốt nhất và bảng kết quả test

Bảng test là cơ sở chính để so sánh mô hình. Cột `FN (bỏ sót M)` đếm số mẫu ác tính bị dự đoán nhầm thành lành tính.

In [ ]:
parameter_rows = []
metric_rows = []

for model_key in model_order:
    result = loaded_results[model_key]
    predictions = result['predictions']
    metrics = result['test_metrics']

    parameter_rows.append({
        'Mô hình': model_info[model_key]['name'],
        'Siêu tham số tốt nhất': ', '.join(
            f"{key.replace('__', ': ')}={value}" for key, value in result['best_parameters'].items()
        ),
    })

    false_negatives = int(((predictions['actual_diagnosis'] == 'M') & (predictions['predicted_diagnosis'] == 'B')).sum())
    metric_rows.append({
        'Mô hình': model_info[model_key]['name'],
        'Accuracy': metrics['accuracy'],
        'Precision M': metrics['precision_malignant'],
        'Recall M': metrics['recall_malignant'],
        'F1 M': metrics['f1_malignant'],
        'F1-macro': metrics['f1_macro'],
        'ROC-AUC': metrics['roc_auc'],
        'FN (bỏ sót M)': false_negatives,
        'Thời gian tuning (giây)': metrics['tuning_seconds'],
        'Thời gian dự đoán test (giây)': metrics['test_prediction_seconds'],
    })

parameters_table = pd.DataFrame(parameter_rows)
test_results = pd.DataFrame(metric_rows)

parameters_table.to_csv(comparison_dir / 'best_parameters_comparison.csv', index=False, encoding='utf-8-sig')
test_results.to_csv(comparison_dir / 'test_metrics_comparison.csv', index=False, encoding='utf-8-sig')

display(parameters_table)
display(test_results.style.format({
    'Accuracy': '{:.2%}', 'Precision M': '{:.2%}', 'Recall M': '{:.2%}',
    'F1 M': '{:.2%}', 'F1-macro': '{:.2%}', 'ROC-AUC': '{:.4f}',
    'Thời gian tuning (giây)': '{:.3f}', 'Thời gian dự đoán test (giây)': '{:.4f}',
}))

## 3. Biểu đồ so sánh chỉ số và thời gian

Các chỉ số chất lượng nằm trong khoảng 0–1 nên có thể đặt chung trong một biểu đồ cột. Thời gian được tách riêng vì dùng đơn vị giây.

In [ ]:
quality_columns = ['Accuracy', 'Recall M', 'F1-macro', 'ROC-AUC']
quality_long = test_results.melt(
    id_vars='Mô hình', value_vars=quality_columns,
    var_name='Chỉ số', value_name='Giá trị',
)

fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=quality_long, x='Chỉ số', y='Giá trị', hue='Mô hình', ax=ax)
ax.set_ylim(0.75, 1.02)
ax.set_title('So sánh các chỉ số trên tập test')
ax.set_ylabel('Giá trị')
ax.legend(title='Mô hình', loc='lower left')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, fontsize=8)
fig.tight_layout()
fig.savefig(comparison_dir / 'test_metrics_bar_chart.png', dpi=200, bbox_inches='tight')
plt.show()

timing_long = test_results.melt(
    id_vars='Mô hình',
    value_vars=['Thời gian tuning (giây)', 'Thời gian dự đoán test (giây)'],
    var_name='Hoạt động', value_name='Giây',
)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=timing_long, x='Mô hình', y='Giây', hue='Hoạt động', ax=ax)
ax.set_title('Thời gian tuning và dự đoán trên tập test')
ax.set_ylabel('Giây')
ax.legend(title='Hoạt động')
fig.tight_layout()
fig.savefig(comparison_dir / 'timing_bar_chart.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Confusion matrix và ROC curve

Ba confusion matrix được vẽ lại từ cùng định dạng dữ liệu dự đoán để dễ đặt cạnh nhau. Với nhãn `B, M`, ô hàng `M`, cột `B` là false negative: ca ác tính bị bỏ sót.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, model_key in zip(axes, model_order):
    predictions = loaded_results[model_key]['predictions']
    matrix = confusion_matrix(
        predictions['actual_diagnosis'], predictions['predicted_diagnosis'], labels=['B', 'M']
    )
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['B (dự đoán)', 'M (dự đoán)'],
                yticklabels=['B (thật)', 'M (thật)'], ax=ax)
    ax.set_title(model_info[model_key]['name'])
    ax.set_xlabel('Nhãn dự đoán')
    ax.set_ylabel('Nhãn thật')
fig.suptitle('Confusion matrix trên cùng tập test', y=1.03, fontsize=14)
fig.tight_layout()
fig.savefig(comparison_dir / 'confusion_matrices_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for model_key in model_order:
    predictions = loaded_results[model_key]['predictions']
    y_true = (predictions['actual_diagnosis'] == 'M').astype(int)
    probabilities = predictions['probability_malignant']
    fpr, tpr, _ = roc_curve(y_true, probabilities)
    model_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2, label=f"{model_info[model_key]['name']} (AUC = {model_auc:.4f})")

ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Đường ngẫu nhiên')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate / Recall M')
ax.set_title('ROC curve của ba mô hình')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(comparison_dir / 'roc_curves_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Ảnh hưởng của tuning trên cross-validation

Bảng này trả lời tuning đã thay đổi từng mô hình như thế nào trên train/CV. Không dùng nó thay thế bảng test vì baseline của từng thuật toán vốn không cùng cấu hình.

In [ ]:
cv_rows = []
for model_key in model_order:
    result = loaded_results[model_key]
    baseline = result['baseline_cv_metrics']
    best_cv = result['cv_results'].sort_values('rank_f1_macro').iloc[0]
    cv_rows.append({
        'Mô hình': model_info[model_key]['name'],
        'Baseline F1-macro CV': baseline['f1_macro']['mean'],
        'Best F1-macro CV': best_cv['mean_f1_macro_cv'],
        'Độ lệch chuẩn Best F1 CV': best_cv['std_f1_macro_cv'],
        'Best Recall M CV': best_cv['mean_recall_malignant_cv'],
        'Chênh lệch F1 CV': best_cv['mean_f1_macro_cv'] - baseline['f1_macro']['mean'],
    })

cv_comparison = pd.DataFrame(cv_rows)
cv_comparison.to_csv(comparison_dir / 'cv_baseline_vs_tuned_comparison.csv', index=False, encoding='utf-8-sig')
display(cv_comparison.style.format({
    'Baseline F1-macro CV': '{:.4f}', 'Best F1-macro CV': '{:.4f}',
    'Độ lệch chuẩn Best F1 CV': '{:.4f}', 'Best Recall M CV': '{:.4f}',
    'Chênh lệch F1 CV': '{:+.4f}',
}))

## 6. Kết luận thực nghiệm

Cell dưới đây tạo một kết luận gợi ý dựa trên Recall lớp M, rồi F1-macro. Nhóm cần dùng kết quả như bằng chứng để giải thích, không chỉ đọc nguyên văn.

In [ ]:
selected_model = test_results.sort_values(['Recall M', 'F1-macro'], ascending=False).iloc[0]
summary_text = (
    f"Trên cùng tập test 114 mẫu, {selected_model['Mô hình']} có Recall lớp M cao nhất "
    f"({selected_model['Recall M']:.2%}) và F1-macro {selected_model['F1-macro']:.4f}. "
    f"Mô hình này bỏ sót {int(selected_model['FN (bỏ sót M)'])}/42 mẫu ác tính trong test."
)
print(summary_text)
print('\nKhuyến nghị báo cáo: ưu tiên diễn giải Recall lớp M và số false negative; không suy rộng kết quả thành hiệu quả chẩn đoán lâm sàng.')
print(f'\nĐã lưu bảng và biểu đồ tổng hợp tại: {comparison_dir}')